# Leaderboard Pipeline 06b — Compare DINOv3 Test Submission with Moondream

This notebook runs Moondream independently on the **test images**, then compares its pseudo-labels with an already-generated DINOv3 submission.

## Interpretation boundary

- Moondream predictions are **not ground truth**.
- Agreement measures consistency between two models, not test accuracy.
- The test images and Moondream outputs must never be used for training, threshold fitting, calibration, or model selection.
- The pipeline does not create an official competition submission.
- Actual test inference is blocked until you explicitly confirm that the competition rules permit external pretrained VLM inference on test images.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from PIL import Image
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "leaderboard_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_ROOT = REPO_ROOT / "BDC2026"
SUBMISSION_PATH = REPO_ROOT / "submission_NamaTim_384.csv"  # change to the file you submitted
DINO_DEBUG_PATH = REPO_ROOT / "test_predictions_debug.csv"  # optional
STAGE_ROOT = REPO_ROOT / "leaderboard_pipeline_outputs" / "06b_moondream_test_comparison"
PREDICTION_DIR = STAGE_ROOT / "moondream_predictions"
COMPARISON_DIR = STAGE_ROOT / "comparison"

MODEL = "moondream3.1-9B-A2B"
BACKEND = "photon"
SMOKE_LIMIT = 5

print("Submission:", SUBMISSION_PATH)
print("Stage root:", STAGE_ROOT)


## 1. Validate alignment without loading Moondream

This confirms that the submitted CSV matches the official template order and that all inferred paths are inside `BDC2026/test`.


In [ ]:
dry_run_command = [
    sys.executable,
    str(REPO_ROOT / "scripts" / "moondream_predict_test.py"),
    "--data-root", str(DATA_ROOT),
    "--submission", str(SUBMISSION_PATH),
    "--output-dir", str(PREDICTION_DIR),
    "--dry-run",
]
print(" ".join(dry_run_command))
subprocess.run(dry_run_command, cwd=REPO_ROOT, check=True)


## 2. Five-image smoke test

Set both flags below only after checking the competition rules. This cell uses a separate output directory so the full run can start cleanly.


In [ ]:
RULES_CONFIRMED = False
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    assert RULES_CONFIRMED, "Confirm the competition rules before test inference."
    smoke_dir = STAGE_ROOT / "smoke_test"
    command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "moondream_predict_test.py"),
        "--data-root", str(DATA_ROOT),
        "--submission", str(SUBMISSION_PATH),
        "--output-dir", str(smoke_dir),
        "--backend", BACKEND,
        "--model", MODEL,
        "--limit", str(SMOKE_LIMIT),
        "--checkpoint-every", "1",
        "--confirm-rules-allow-external-vlm-test-inference",
    ]
    if BACKEND == "transformers":
        command.append("--compile")
    print(" ".join(command))
    subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("Smoke test disabled.")


## 3. Full test inference

The script checkpoints every five images and resumes from `moondream_test_predictions.csv`. For the preview model, set `BACKEND="transformers"` and `MODEL="moondream/moondream3-preview"`.


In [ ]:
RUN_FULL_INFERENCE = False

if RUN_FULL_INFERENCE:
    assert RULES_CONFIRMED, "Confirm the competition rules before test inference."
    command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "moondream_predict_test.py"),
        "--data-root", str(DATA_ROOT),
        "--submission", str(SUBMISSION_PATH),
        "--output-dir", str(PREDICTION_DIR),
        "--backend", BACKEND,
        "--model", MODEL,
        "--limit", "0",
        "--checkpoint-every", "5",
        "--confirm-rules-allow-external-vlm-test-inference",
    ]
    if BACKEND == "transformers":
        command.append("--compile")
    print(" ".join(command))
    subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("Full inference disabled.")


## 4. Generate comparison reports

DINO probabilities are optional. When `test_predictions_debug.csv` is available, the report also ranks disagreements by DINO confidence and margin.


In [ ]:
predictions_path = PREDICTION_DIR / "moondream_test_predictions.csv"
RUN_COMPARISON = predictions_path.exists()

if RUN_COMPARISON:
    command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "compare_submission_moondream.py"),
        "--moondream-predictions", str(predictions_path),
        "--output-dir", str(COMPARISON_DIR),
    ]
    if DINO_DEBUG_PATH.exists():
        command.extend(["--dino-debug", str(DINO_DEBUG_PATH)])
    print(" ".join(command))
    subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("Run Moondream inference first:", predictions_path)


## 5. Inspect agreement and disagreement

Focus on high-confidence, non-ambiguous disagreements as **review cases**, not presumed corrections.


In [ ]:
summary_path = COMPARISON_DIR / "comparison_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    display(pd.Series(summary).to_frame("value"))

matrix_path = COMPARISON_DIR / "agreement_matrix.csv"
if matrix_path.exists():
    display(pd.read_csv(matrix_path, index_col=0))

distribution_path = COMPARISON_DIR / "class_distributions.csv"
if distribution_path.exists():
    display(pd.read_csv(distribution_path))


In [ ]:
disagreement_path = COMPARISON_DIR / "high_confidence_unambiguous_disagreements.csv"
if disagreement_path.exists():
    disagreements = pd.read_csv(disagreement_path)
    columns = [
        column for column in [
            "test_index", "resolved_path", "submission_predicted",
            "moondream_suggested_label", "moondream_confidence",
            "dino_confidence", "dino_margin", "moondream_reason"
        ] if column in disagreements.columns
    ]
    display(disagreements[columns].head(50))

image_path = COMPARISON_DIR / "disagreement_grid.png"
if image_path.exists():
    display(Image.open(image_path))


## Decision rule for the two remaining submissions

Do not submit a Moondream-only CSV. Use this comparison only to understand disagreement patterns.

A second competition submission should still be selected from **training OOF evidence**. Moondream test agreement may be recorded as a diagnostic, but it must not become the optimization target because the hidden labels are unavailable and Moondream can share systematic semantic mistakes with DINOv3.
